<a href="https://colab.research.google.com/github/FC-Andrade/Analises-complementares/blob/main/AN%C3%81LISE_DE_FLEXIBILIDADE_DO_PEPT%C3%8DDEO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================
# 🔬 ANÁLISE DE FLEXIBILIDADE DO PEPTÍDEO AG73
# COMPLEXO COM O SYNDECAN-4
# ==========================
# Este notebook calcula RMSD, RMSF e gera figuras equivalentes à Figura 9B
# Adaptado para uso direto no Google Colab

# --- Instalação e importação das bibliotecas ---
!pip install -q MDAnalysis matplotlib numpy seaborn nglview

import MDAnalysis as mda
from MDAnalysis.analysis import rms, align
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import nglview as nv
from google.colab import files

sns.set(style="whitegrid", font_scale=1.2)



In [ ]:
# --- Upload dos arquivos ---
print("📁 Faça o upload dos arquivos de topologia (.pdb ou .gro) e trajetórias (.xtc) das réplicas:")
uploaded = files.upload()

# Exemplo esperado:
#   replica1.pdb
#   replica1.xtc
#   replica2.pdb
#   replica2.xtc
#   replica3.pdb
#   replica3.xtc



In [ ]:
# --- Defina o nome do residuo do peptídeo ---
ligand_resname = "AG73"  # altere se necessário

# --- Função para calcular RMSD e RMSF ---
def analyze_trajectory(top_file, traj_file, ligand_resname):
    u = mda.Universe(top_file, traj_file)
    ligand = u.select_atoms(f"resname {ligand_resname}")

    # Alinhar a trajetória ao primeiro frame
    aligner = align.AlignTraj(u, u, select=f"resname {ligand_resname}", in_memory=True)
    aligner.run()

    # RMSD
    R = rms.RMSD(ligand, ligand, ref_frame=0)
    R.run()
    rmsd = R.rmsd[:, 2]

    # RMSF
    avg_positions = np.mean(ligand.positions, axis=0)
    displacements = []
    for ts in u.trajectory:
        displacements.append(np.sqrt(((ligand.positions - avg_positions) ** 2).sum(axis=1)))
    rmsf = np.mean(displacements, axis=0)

    return rmsd, rmsf



In [ ]:
# --- Processar as três réplicas ---
replicas = [("replica1.pdb", "replica1.xtc"),
            ("replica2.pdb", "replica2.xtc"),
            ("replica3.pdb", "replica3.xtc")]

all_rmsd, all_rmsf = [], []
for top, traj in replicas:
    rmsd, rmsf = analyze_trajectory(top, traj, ligand_resname)
    all_rmsd.append(rmsd)
    all_rmsf.append(rmsf)

# --- Calcular média e desvio padrão ---
rmsd_mean = np.mean(np.vstack(all_rmsd), axis=0)
rmsd_std = np.std(np.vstack(all_rmsd), axis=0)
rmsf_mean = np.mean(np.vstack(all_rmsf), axis=0)
rmsf_std = np.std(np.vstack(all_rmsf), axis=0)

# --- Plot RMSD e RMSF ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# RMSD
sns.histplot(np.concatenate(all_rmsd), color="green", kde=True, ax=axes[0], alpha=0.5)
axes[0].set_xlabel("RMSD (nm)")
axes[0].set_ylabel("Counts (a.u.)")
axes[0].set_title("Distribuição RMSD – AG73/Syndecan-4")

# RMSF
residues = np.arange(1, len(rmsf_mean)+1)
axes[1].plot(residues, rmsf_mean, color="green", lw=2)
axes[1].fill_between(residues, rmsf_mean - rmsf_std, rmsf_mean + rmsf_std,
                     color="green", alpha=0.3)
axes[1].set_xlabel("Resíduo")
axes[1].set_ylabel("RMSF (nm)")
axes[1].set_title("RMSF por resíduo – AG73/Syndecan-4")

plt.tight_layout()
plt.show()



In [ ]:
# --- Salvar figuras em alta resolução ---
plt.savefig("C16_avb3_RMSD_RMSF.png", dpi=600, bbox_inches='tight')
plt.show()

# --- Salvar dados em CSV ---
df_rmsd = pd.DataFrame(np.vstack(all_rmsd).T, columns=["Réplica1", "Réplica2", "Réplica3"])
df_rmsf = pd.DataFrame({"Resíduo": residues,
                        "RMSF_médio": rmsf_mean,
                        "Desvio_Padrão": rmsf_std})

df_rmsd.to_csv("AG73-Syndecan-4_RMSD.csv", index=False)
df_rmsf.to_csv("AG73-Syndecan-4_RMSF.csv", index=False)

print("📊 Arquivos gerados:")
print("- AG73-Syndecan-4_RMSD_RMSF.png")
print("- AG73-Syndecan-4_RMSD.csv")
print("- AG73-Syndecan-4_RMSF.csv")

# --- Download automático ---
files.download("AG73-Syndecan-4_RMSD_RMSF.png")
files.download("AG73-Syndecan-4_RMSD.csv")
files.download("AG73-Syndecan-4_RMSF.csv")

# --- Visualização 3D opcional ---
print("🔍 Visualização 3D do ligante no frame final:")
u = mda.Universe("replica1.pdb", "replica1.xtc")
view = nv.show_mdanalysis(u)
view.add_representation("cartoon", selection="protein")
view.add_representation("licorice", selection=f"resname {ligand_resname}", color="green")
view
